# Pipeline Connections: deep-framex -> BIIGLE

Use this when you already have `deep-framex` outputs and want to push them to BIIGLE through the REST API.

Expected inputs:
- a BIIGLE-accessible frame location (hosted URL or local-instance storage path)
- a metadata CSV (typically `frames/biigle_metadata.csv`) with a required `filename` column

API flow:
1. `POST /api/v1/projects` (create project)
2. `POST /api/v1/projects/{project_id}/pending-volumes` (upload metadata CSV)
3. `PUT /api/v1/pending-volumes/{pending_id}` (finalize real volume with file list + base URL)


## Local BIIGLE

If you run your own BIIGLE instance, you have two common options:
- Use BIIGLE-managed files (Manual: **Files -> Storage requests**) to upload files to your BIIGLE server UI.
- Use a local/instance storage location and set `IMAGE_BASE_URL` to a BIIGLE-accessible path/URL.

For this notebook's API flow, set:
- `BIIGLE_URL` to your local instance URL (for example `http://localhost:8080`)
- `IMAGE_BASE_URL` to the location your BIIGLE instance can serve from

## Online-hosted images

Use this mode when your frames are already hosted (S3/HTTP/static web host) and BIIGLE should reference them.

Set:
- `BIIGLE_URL` to the BIIGLE instance you use
- `IMAGE_BASE_URL` to the hosted prefix, so BIIGLE resolves each file as:
  - `<IMAGE_BASE_URL>/<filename>`

From the BIIGLE manual (Remote locations): hosted files should be reachable by browser and BIIGLE, and CORS must allow your BIIGLE origin.


In [ ]:
import csv
from pathlib import Path
import requests

# Fill these in:
BIIGLE_URL = "https://biigle.de"
EMAIL = "you@example.org"
API_TOKEN = "your_api_token_here"

PROJECT_NAME = "PROJECT"
PROJECT_DESC = "deep-framex API upload test"
VOLUME_NAME = "VOLUME"

METADATA_CSV = Path("frames/biigle_metadata.csv")
IMAGE_BASE_URL = "https://image/base/url"
METADATA_PARSER = "Biigle\\Services\\MetadataParsing\\ImageCsvParser"

# 1) Read filenames from metadata CSV
with METADATA_CSV.open("r", newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

filenames = [r["filename"] for r in rows if r.get("filename")]
print(f"Found {len(filenames)} files in metadata CSV")

# 2) Open BIIGLE API session (HTTP Basic auth with email + token)
session = requests.Session()
session.auth = (EMAIL, API_TOKEN)
session.headers.update({"Accept": "application/json"})

# 3) Create project
project_resp = session.post(
    f"{BIIGLE_URL}/api/v1/projects",
    json={"name": PROJECT_NAME, "description": PROJECT_DESC},
    timeout=120,
)
if not project_resp.ok:
    raise RuntimeError(f"Create project failed: HTTP {project_resp.status_code} - {project_resp.text}")

project_id = project_resp.json()["id"]
print("Project created:", project_id)

# 4) Create pending volume and upload metadata CSV
with METADATA_CSV.open("rb") as fp:
    pending_resp = session.post(
        f"{BIIGLE_URL}/api/v1/projects/{project_id}/pending-volumes",
        data={"media_type": "image", "metadata_parser": METADATA_PARSER},
        files={"metadata_file": (METADATA_CSV.name, fp, "text/csv")},
        timeout=120,
    )

if not pending_resp.ok:
    raise RuntimeError(f"Create pending volume failed: HTTP {pending_resp.status_code} - {pending_resp.text}")

pending_id = pending_resp.json()["id"]
print("Pending volume created:", pending_id)


In [ ]:
# 5) Finalize pending volume into a real BIIGLE volume
finalize_resp = session.put(
    f"{BIIGLE_URL}/api/v1/pending-volumes/{pending_id}",
    json={"name": VOLUME_NAME, "url": IMAGE_BASE_URL, "files": filenames},
    timeout=120,
)

if not finalize_resp.ok:
    raise RuntimeError(f"Finalize volume failed: HTTP {finalize_resp.status_code} - {finalize_resp.text}")

volume_id = finalize_resp.json()["volume_id"]
print("Volume created:", volume_id)
print({"project_id": project_id, "pending_id": pending_id, "volume_id": volume_id})


# Pipeline Connections: deep-framex -> MARIMBA

This notebook shows a practical handoff path for regular users:

1. Start with `deep-framex` outputs (`*.jpg`, `ifdo.json`, `biigle_metadata.csv`, though the BIIGLE output is not required here)
2. Import those outputs into a MARIMBA collection using `marimba import`
3. Optionally run `marimba process` and `marimba package`

This is intentionally minimal and focuses on interoperability, not full MARIMBA pipeline development.

## What Is the Entry Point into MARIMBA?

For ingesting existing data products (like `deep-framex` frames), the entrypoint is:

- `marimba import <collection_name> <source_path> ...`

Under the hood this calls each installed MARIMBA pipeline's `_import()` implementation and writes data into the collection's pipeline-specific directory:

- `collections/<collection_name>/<pipeline_name>/...`


## Setup

Edit these paths and names for your environment.

In [ ]:
from pathlib import Path

# deep-framex output directory
deep_framex_output = Path("frames")

# Existing MARIMBA project root (contains pipelines/, collections/, datasets/)
marimba_project_dir = Path("marimba-project")

# MARIMBA target names
collection_name = "dfx-collection-001"
pipeline_name = "my-instrument"  # must match an installed MARIMBA pipeline
dataset_name = "dfx-dataset-001"

print('deep_framex_output =', deep_framex_output.resolve())
print('marimba_project_dir =', marimba_project_dir.resolve())

## Validate deep-framex Outputs

We expect at least JPEG frames and usually iFDO sidecars:

- `ifdo.json`

In [ ]:
jpgs = sorted(deep_framex_output.glob('*.jpg'))
ifdo_path = deep_framex_output / 'ifdo.json'

print(f'Found {len(jpgs)} JPEG frames')
print('ifdo.json exists:', ifdo_path.exists())

if len(jpgs) == 0:
    raise FileNotFoundError(f'No JPEGs found in {deep_framex_output}')

## Build `marimba import` Command

We use `--operation link` by default to avoid duplicating large frame sets.

Use `--operation copy` if you want MARIMBA to own independent copies.

In [ ]:
import shlex

marimba_import_cmd = [
    'marimba',
    'import',
    collection_name,
    str(deep_framex_output),
    '--project-dir',
    str(marimba_project_dir),
    '--pipeline-name',
    pipeline_name,
    '--operation',
    'link',
]

print('\n'.join(['Command:'] + ['  ' + shlex.join(marimba_import_cmd)]))

## Run `marimba import`

Set `dry_run=True` first if you want to preview behavior before writing changes.

In [ ]:
import subprocess

dry_run = True
cmd = marimba_import_cmd + (['--dry-run'] if dry_run else [])

print('Running:', shlex.join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print('exit code:', result.returncode)
print('--- stdout ---')
print(result.stdout)
print('--- stderr ---')
print(result.stderr)

if result.returncode != 0:
    raise RuntimeError('marimba import failed; inspect output above')

## Inspect Where Data Lands

After a real (non-dry-run) import, data is expected under:

- `collections/<collection_name>/<pipeline_name>/`

In [ ]:
pipeline_data_dir = marimba_project_dir / 'collections' / collection_name / pipeline_name
print('pipeline_data_dir =', pipeline_data_dir)

if pipeline_data_dir.exists():
    sample = sorted(pipeline_data_dir.rglob('*'))[:25]
    print(f'Showing first {len(sample)} entries:')
    for p in sample:
        print(' -', p)
else:
    print('Directory does not exist yet (expected if dry_run=True).')

## About iFDO in This Handoff

`deep-framex` produces an `ifdo.json` sidecar for the extracted frames. In this interoperability pattern:

- The sidecar can be imported alongside JPEGs as provenance/context.
- MARIMBA dataset-level metadata output is still driven by your MARIMBA pipeline's `_package()` mapping logic.

So this notebook demonstrates data ingress into MARIMBA. Whether and how imported iFDO is merged into packaged MARIMBA metadata depends on the specific MARIMBA pipeline implementation.

## Optional Next Commands

If your MARIMBA pipeline defines processing and packaging behavior, these are the next stages:

```bash
marimba process --project-dir <project> --collection-name <collection> --pipeline-name <pipeline>
marimba package <dataset> --project-dir <project> --collection-name <collection> --pipeline-name <pipeline> --version 1.0 --contact-name "..." --contact-email "..."
```